# Respiratory failure: revision analyses for IBMED-D-26-00792

Standalone notebook. The engine (GLOBAL_STATS, FEATURE_WEIGHTS, STATE_Z, STATE_SCALE, `_age_params`,
`resp_state_estimation_patient`, momentum rule) is imported **verbatim** from cell 1 of `RF-PEMDT.ipynb`
with its `if __name__ == "__main__"` block removed, so the full MIMIC pipeline is NOT re-run.
Input: `RF_PEMDT_output/RF_cohort_windows.csv` (the windows the published model was run on).

| Section | Reviewer point |
|---|---|
| R1 calibration under the length-fair pre-onset protocol (intercept, slope, Brier, reliability) | R1.4, R2.3 |
| R2 operating characteristics; false alerts per patient-day in never-ventilated patients | R1.5, R2.2 |
| R3 ablation (momentum, ordering, weights, age, dynamics) | R1.3 |
| R4 baselines (LR, RF, gradient boosting, GRU) on identical windows and patient split | R1.2 |
| R5 temporal validation by anchor_year_group | R1.10 |
| R6 attribution fidelity and stability | R1.7 |
| R7 computational cost and reproducibility record | R1.9, R2.1 |

Version 2: time windows are computed independently of the pandas datetime unit, alert burden
uses monitored stay time, and the published pre-onset results are checked before anything else.
Outputs: `results/revision_v2/` (git-ignored; MIMIC-derived, do not commit).
Expected run time on a laptop: about 30 to 60 minutes; peak memory about 4 to 6 GB.

In [1]:
# ---------------------------------------------------------------- R0 · setup
import os, sys, time, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd().parent))
import revision_utils as ru

SRC_NB = Path("RF-PEMDT.ipynb")
exec(ru.strip_main_guard(ru.cell_source(SRC_NB, "Respiratory-Failure Digital Twin")), globals())

REV = Path("results/revision_v2"); REV.mkdir(parents=True, exist_ok=True)
FIGR = REV / "figures"; FIGR.mkdir(exist_ok=True)
SYNTHETIC_TEST = os.environ.get("REV_SYNTHETIC", "0") == "1"
N_BOOT = int(os.environ.get("REV_BOOT", "200"))
SEED = 0                                   # cell 9/10 seed
HORIZONS_H = [0, 2, 4, 6, 12, 24]          # as published
W_MIN = 4 * 60                             # length-fair window (minutes), as published
FEATS = list(LIK_FEATURES)
print("Environment:", ru.env_versions())

Environment: {'python': '3.13.5', 'platform': 'macOS-27.0.1-arm64-arm-64bit-Mach-O', 'numpy': '2.5.4', 'pandas': '2.3.3', 'scipy': '1.18.1', 'sklearn': '1.9.1', 'jax': '0.10.0', 'xgboost': '3.2.0', 'torch': '2.13.0', 'lifelines': '0.30.3'}


In [2]:
# -------------------------------------------------- R0b · load the modelled windows
USE = ["patient_id", "subject_id", "stay_id", "timestamp", "anchor_age", "first_vent_time", "has_vent",
       "is_resp_failure"] + FEATS
if SYNTHETIC_TEST:
    rng = np.random.default_rng(1); rows = []
    for i in range(int(os.environ.get("REV_NPAT", "800"))):
        n = int(rng.integers(150, 900)); pos = rng.random() < 0.45; vent = pos and rng.random() < 0.8
        t0 = pd.Timestamp("2150-01-01") + pd.Timedelta(days=int(rng.integers(0, 3000)))
        ts = t0 + pd.to_timedelta(np.arange(n) * 5, unit="min")
        sev = np.zeros(n)
        if vent:
            k = int(rng.integers(n // 3, n)); sev[max(0, k - 150):k] = np.linspace(0, 2, k - max(0, k - 150))
        df = pd.DataFrame({"patient_id": f"p{1000+i}", "subject_id": 1000 + i, "stay_id": 5000 + i, "timestamp": ts,
                           "anchor_age": int(rng.integers(20, 95)),
                           "first_vent_time": ts[k] if vent else pd.NaT, "has_vent": int(vent),
                           "is_resp_failure": int(pos)})
        for f in FEATS:
            mu, sd = GLOBAL_STATS[f]
            v = mu + sd * (np.array(STATE_Z[f])[np.clip(np.round(sev).astype(int), 0, 2)] * 0.6
                           + rng.normal(0, 1, n))
            v[rng.random(n) < (0.2 if f in ("SpO2", "HR_bpm", "Resp_rate", "MAP") else 0.95)] = np.nan
            df[f] = v
        rows.append(df)
    W = pd.concat(rows, ignore_index=True)
else:
    W = pd.read_csv(out("RF_cohort_windows.csv"), usecols=USE, low_memory=False)
W["timestamp"] = pd.to_datetime(W["timestamp"], errors="coerce")
W["first_vent_time"] = pd.to_datetime(W["first_vent_time"], errors="coerce")
W[FEATS] = W[FEATS].astype("float64")       # keep float64: identical arithmetic to the published engine
W = W.sort_values(["patient_id", "timestamp"], kind="stable").reset_index(drop=True)
# minutes since 1970, independent of the datetime unit (pandas 3 stores microseconds, pandas 2
# nanoseconds; version 1 assumed nanoseconds and made every window 1,000 times too long)
W["tmin"] = (W["timestamp"] - pd.Timestamp(0)).dt.total_seconds().to_numpy() / 60.0
W["stay_id"] = W["stay_id"].fillna(-1)
print(f"windows {len(W):,} | patients {W.patient_id.nunique():,} | RF prevalence "
      f"{W.groupby('patient_id').is_resp_failure.first().mean():.3f}")

_pid = W["patient_id"].to_numpy()
STARTS = np.r_[0, np.nonzero(_pid[1:] != _pid[:-1])[0] + 1]
LENS = np.diff(np.r_[STARTS, len(W)])
PAT = W["patient_id"].to_numpy()[STARTS]
AGE = W["anchor_age"].to_numpy(float)[STARTS]
AGE = np.where(np.isfinite(AGE) & (AGE > 18) & (AGE < 110), AGE, 65)   # same rule as the engine
PRIORS = np.stack([_age_params(a)[0] for a in AGE]); TBANK = np.stack([_age_params(a)[1] for a in AGE])
ROW_PAT = np.repeat(np.arange(len(STARTS)), LENS)

# patient split for anything that is fitted (Platt, baselines): 30% of patients, seed 0
_rng = np.random.default_rng(SEED)
_perm = _rng.permutation(len(PAT))
TEST_PAT = set(PAT[_perm[: int(0.30 * len(PAT))]])

windows 9,987,192 | patients 65,363 | RF prevalence 0.539


In [3]:
# ------------------ R0c · vectorised forward filter (all variants) + equivalence check
MOM_ROW = np.array([0.02, 0.08, 0.90])


def unordered(A):
    K = A.shape[-1]; U = np.empty_like(A)
    for k in range(K):
        U[..., k, :] = ((1 - A[..., k, k]) / (K - 1))[..., None]; U[..., k, k] = A[..., k, k]
    return U


def loglik_matrix(weights=None, resp_gain=1.0, noise=None, drop=None):
    """(N,3) weighted Gaussian log-likelihood in z-space, identical to the engine."""
    log2pi = np.log(2 * np.pi); L = np.zeros((len(W), 3))
    for f in FEATS:
        if drop is not None and f == drop: continue
        v = W[f].to_numpy(float); ok = ~np.isnan(v)
        mu, sd = GLOBAL_STATS[f]; z = (v - mu) / sd
        w = (weights or FEATURE_WEIGHTS)[f] * (resp_gain if f in RESP_CORE else 1.0)
        for s in range(3):
            loc, sc = STATE_Z[f][s], STATE_SCALE[s]
            L[ok, s] += (w * (-0.5 * ((z - loc) / sc) ** 2 - np.log(sc) - 0.5 * log2pi))[ok]
    return L


def filter_flat(logL, priors, tbank, momentum=True, static=False, unordered_T=False, chunk=4096):
    """Engine recursion: belief starts at the age prior; at every window (including the first)
    predict with T (momentum row if belief[Failure] > 0.4), update, renormalise."""
    out = np.empty((len(logL), 3))
    order = np.argsort(LENS, kind="stable")
    mom_bank = tbank.copy(); mom_bank[:, 2, :] = MOM_ROW
    if unordered_T:
        mom_bank = unordered(mom_bank)
    for ch in np.array_split(order, max(1, len(order) // chunk)):
        st, ln = STARTS[ch], LENS[ch]
        b = priors[ch].copy(); Tb, Tm, pr = tbank[ch], mom_bank[ch], priors[ch]
        for t in range(int(ln.max())):
            a0 = np.searchsorted(ln, t, side="right")
            bb = b[a0:]
            if static:
                pred = pr[a0:]
            else:
                T = np.where((momentum & (bb[:, 2] > 0.4))[:, None, None], Tm[a0:], Tb[a0:])
                pred = np.einsum("bk,bkj->bj", bb, T)
            lp = np.log(pred + 1e-300) + logL[st[a0:] + t]
            lp -= lp.max(1, keepdims=True); p = np.exp(lp); p /= p.sum(1, keepdims=True) + 1e-12
            b[a0:] = p; out[st[a0:] + t] = p
    return out


VARIANTS = {
    "Full model":                    {},
    "No failure momentum":           dict(momentum=False),
    "Equal feature weights":         dict(weights={f: 1.0 for f in FEATS}),
    "Unordered transitions":         dict(unordered=True),
    "No age-specific prior/dynamics": dict(no_age=True),
    "No temporal dynamics (static)": dict(static=True),
}
def run_variant(name, return_belief=False):
    cfg = VARIANTS[name]
    pri, tb = PRIORS, TBANK
    if cfg.get("no_age"):
        p50, t50 = _age_params(50.0); pri = np.tile(p50, (len(PAT), 1)); tb = np.tile(t50, (len(PAT), 1, 1))
    un = cfg.get("unordered", False)
    if un:
        tb = unordered(tb)
    B = filter_flat(loglik_matrix(cfg.get("weights")), pri, tb, momentum=cfg.get("momentum", True),
                    static=cfg.get("static", False), unordered_T=un)
    return B if return_belief else B[:, 2].astype(np.float32)


t0 = time.time()
LOGL_FULL = loglik_matrix()
B_FULL = run_variant("Full model", return_belief=True)
T_NUMPY = time.time() - t0
FP = {"Full model": B_FULL[:, 2].astype(np.float32)}
print(f"numpy filter: {len(W):,} windows in {T_NUMPY:.1f} s")

# equivalence with the published per-patient engine
chk = np.random.default_rng(SEED).choice(len(PAT), size=min(100, len(PAT)), replace=False)
t0 = time.time(); diffs = []; nwin = 0; n_dup = 0
for k in chk:
    g = W.iloc[STARTS[k]:STARTS[k] + LENS[k]]
    if g["timestamp"].duplicated().any():          # tied timestamps: row order not identifiable
        n_dup += 1; continue
    r = resp_state_estimation_patient(g.copy())
    diffs.append(np.max(np.abs(r["failure_probability"].to_numpy() - B_FULL[STARTS[k]:STARTS[k] + LENS[k], 2])))
    nwin += LENS[k]
T_LOOP_PER_WIN = (time.time() - t0) / max(nwin, 1)
MAXDIFF = float(np.max(diffs))
print(f"max |numpy - published engine| over {len(diffs)} patients = {MAXDIFF:.2e} "
      f"({n_dup} sampled patients skipped for tied timestamps)")
assert MAXDIFF < 1e-6, "vectorised filter does not reproduce the published engine; stop and check"

numpy filter: 9,987,192 windows in 5.0 s
max |numpy - published engine| over 98 patients = 0.00e+00 (2 sampled patients skipped for tied timestamps)


In [4]:
# ----------------------------- R0d · length-fair pre-onset windows (exactly as cell 9)
neg = W.loc[W.is_resp_failure == 0, ["patient_id", "tmin"]]
span = neg.groupby("patient_id")["tmin"].agg(["min", "max"])
_r = np.random.default_rng(SEED)
span["pt"] = _r.uniform(span["min"].to_numpy(), np.nextafter(span["max"].to_numpy(), np.inf))
PT = W["patient_id"].map(span["pt"]).to_numpy()              # NaN for non-negatives
MB = ((W["first_vent_time"] - W["timestamp"]).dt.total_seconds() / 60.0).to_numpy()
IS_POS = ((W.is_resp_failure == 1) & (W.has_vent == 1) & W.first_vent_time.notna()).to_numpy()
IS_NEG = (W.is_resp_failure == 0).to_numpy()
n_all_pos = W.loc[W.is_resp_failure == 1, "patient_id"].nunique()
print(f"positives {n_all_pos} | with ventilation time {W.loc[IS_POS,'patient_id'].nunique()} | "
      f"negatives {len(span)}")


def window_rows(H, width=W_MIN):
    """Row indices of the length-fair window for horizon H (positives) and the matched
    random-time window (negatives); `width` > W_MIN gives a longer history context."""
    pos = IS_POS & (MB >= H * 60) & (MB <= H * 60 + width)
    ng = IS_NEG & (W["tmin"].to_numpy() <= PT) & (W["tmin"].to_numpy() >= PT - width)
    return np.nonzero(pos)[0], np.nonzero(ng)[0]


def window_scores(fp, H):
    """Per-patient peak failure probability in the window; label 1 = ventilated."""
    pr, nr = window_rows(H)
    s = pd.concat([pd.Series(fp[pr]).groupby(W.patient_id.to_numpy()[pr]).max().rename("score").to_frame().assign(y=1),
                   pd.Series(fp[nr]).groupby(W.patient_id.to_numpy()[nr]).max().rename("score").to_frame().assign(y=0)])
    s.index.name = "patient_id"
    s["is_test"] = s.index.isin(TEST_PAT)
    return s.reset_index()


# reproduce the published horizon AUROCs (all patients, raw score)
from sklearn.metrics import roc_auc_score
WS = {H: window_scores(FP["Full model"], H) for H in HORIZONS_H}
_pub = out("validation_horizon/horizon_metrics.csv")
PUB = pd.read_csv(_pub).set_index("horizon_h") if (os.path.exists(_pub) and not SYNTHETIC_TEST) else None
for H in HORIZONS_H:
    npos, nneg = int(WS[H].y.sum()), int((WS[H].y == 0).sum())
    auc = roc_auc_score(WS[H].y, WS[H].score)
    msg = f"  H={H:>2} h: n_pos={npos:>6}  n_neg={nneg:>6}  AUROC={auc:.4f}"
    if PUB is not None and H in PUB.index:
        msg += f"   | published n_pos={int(PUB.loc[H, 'n_pos'])} n_neg={int(PUB.loc[H, 'n_neg'])} AUROC={PUB.loc[H, 'auc']:.4f}"
        print(msg)
        assert (npos, nneg) == (int(PUB.loc[H, "n_pos"]), int(PUB.loc[H, "n_neg"])) and abs(auc - PUB.loc[H, "auc"]) < 1e-4, \
            "pre-onset windows do not reproduce the published horizon_metrics.csv; stop and send this output"
    else:
        print(msg)
print("Published pre-onset horizon results reproduced." if PUB is not None else "(published horizon file not checked)")

positives 35223 | with ventilation time 27841 | negatives 30140
  H= 0 h: n_pos= 26483  n_neg= 26531  AUROC=0.6505   | published n_pos=26483 n_neg=26531 AUROC=0.6505
  H= 2 h: n_pos= 11595  n_neg= 26531  AUROC=0.6374   | published n_pos=11595 n_neg=26531 AUROC=0.6374
  H= 4 h: n_pos=  6727  n_neg= 26531  AUROC=0.6421   | published n_pos=6727 n_neg=26531 AUROC=0.6421
  H= 6 h: n_pos=  4669  n_neg= 26531  AUROC=0.6593   | published n_pos=4669 n_neg=26531 AUROC=0.6593
  H=12 h: n_pos=  3369  n_neg= 26531  AUROC=0.6453   | published n_pos=3369 n_neg=26531 AUROC=0.6453
  H=24 h: n_pos=  2234  n_neg= 26531  AUROC=0.6275   | published n_pos=2234 n_neg=26531 AUROC=0.6275
Published pre-onset horizon results reproduced.


In [5]:
# ------------- R1-audit · resolution of the pre-onset window score (decides the recalibration)
# Paste after R0d. Uses WS, B_FULL, window_scores, HORIZONS_H, REV, ru. Refits only the paper's
# Platt map on TRAINING patients, never looks at test-patient outcomes, writes two small CSVs.
from sklearn.metrics import roc_auc_score


def _logit(v):
    v = np.asarray(v, float)
    return np.log(v) - np.log1p(-v)


def _group_codes(s):
    """0 = exact 0, 1..10 = deciles of log-odds among 0 < s < 1, 11 = exact 1."""
    code = np.zeros(len(s), int)
    mid = (s > 0) & (s < 1)
    if mid.any():
        code[mid] = pd.qcut(_logit(s[mid]), 10, labels=False, duplicates="drop") + 1
    code[s >= 1] = 11
    return code


_lab = {0: "exact 0", 11: "exact 1"}
aud, dec = [], []
for H in HORIZONS_H:
    d = WS[H]                                                     # published float32 window peak
    s64 = (window_scores(B_FULL[:, 2], H).set_index("patient_id")["score"]
           .reindex(d["patient_id"]).to_numpy(float))             # same windows, float64 belief
    tr = ~d["is_test"].to_numpy(); y = d["y"].to_numpy()
    for prec, s in [("float32 (published)", d["score"].to_numpy(float)), ("float64", s64)]:
        vals, cnt = np.unique(s, return_counts=True)
        aud.append(dict(horizon_h=H, precision=prec, patients=len(s),
                        exact_zero=np.mean(s == 0), below_1e_8=np.mean(s < 1e-8),
                        above_1_minus_1e_7=np.mean(s > 1 - 1e-7), exact_one=np.mean(s == 1),
                        distinct_values=len(vals), largest_tie_share=cnt.max() / len(s),
                        value_of_largest_tie=float(vals[cnt.argmax()]),
                        train_auroc=roc_auc_score(y[tr], s[tr])))
    st, yt = d["score"].to_numpy(float)[tr], y[tr]
    p_now = ru.platt_apply(ru.platt_fit(st, yt), st)                 # the map used in the paper
    t = (pd.DataFrame({"g": _group_codes(st), "s": st, "y": yt, "p": p_now})
         .groupby("g").agg(patients=("y", "size"), raw_median=("s", "median"),
                           current_prob=("p", "mean"), observed=("y", "mean")))
    t["share"] = t["patients"] / len(st)
    t.index = [_lab.get(k, f"log-odds decile {k}") for k in t.index]
    dec.append(t.reset_index(names="group").assign(horizon_h=H))

AUD = pd.DataFrame(aud)
DEC = pd.concat(dec, ignore_index=True)[["horizon_h", "group", "patients", "share", "raw_median",
                                         "current_prob", "observed"]]
AUD.to_csv(REV / "R1_audit_score_resolution.csv", index=False)
DEC.to_csv(REV / "R1_audit_training_groups.csv", index=False)
pd.set_option("display.width", 200)
print("Window-score resolution, all patients (training AUROC only uses training outcomes):")
print(AUD.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print("\nTraining patients only: observed rate by group vs the current Platt probability:")
print(DEC.to_string(index=False, float_format=lambda x: f"{x:.4g}"))

Window-score resolution, all patients (training AUROC only uses training outcomes):
 horizon_h           precision  patients  exact_zero  below_1e_8  above_1_minus_1e_7  exact_one  distinct_values  largest_tie_share  value_of_largest_tie  train_auroc
         0 float32 (published)     53014           0   0.0002641            0.005093   0.004452            48349           0.004452                     1       0.6502
         0             float64     53014           0   0.0002641            0.005093          0            48746           0.002848              0.001025       0.6502
         2 float32 (published)     38126           0   0.0002885            0.002256   0.001967            36715           0.001967                     1       0.6371
         2             float64     38126           0   0.0002885            0.002256          0            36820           0.000577                0.0371       0.6371
         4 float32 (published)     33258           0   0.0003007            0.002

In [6]:
# ------------------------- R1 · calibration under the pre-onset protocol (test patients)
rows, PLATT = [], {}
for H in HORIZONS_H:
    d = WS[H]; tr, te = d[~d.is_test], d[d.is_test]
    PLATT[H] = ru.platt_fit(tr.score, tr.y)
    d["p_cal"] = ru.platt_apply(PLATT[H], d.score)
    for nm, col in [("raw window peak", "score"), ("recalibrated (Platt, train)", "p_cal")]:
        rows.append(ru.evaluate(te.y, d.loc[d.is_test, col], te.patient_id, n_boot=N_BOOT, seed=SEED,
                                label=f"{H:>2} h | {nm}", rank_score=te.score))
R1 = pd.concat(rows, ignore_index=True)
# concurrent (published Figure 1): out-of-fold calibrated patient probabilities from cell 4
_cp = out("validation_calibrated/calibrated_patients.csv")
if os.path.exists(_cp) and not SYNTHETIC_TEST:
    cp = pd.read_csv(_cp)
    R1 = pd.concat([R1, ru.evaluate(cp.label, cp.calibrated_prob, cp.patient_id, n_boot=N_BOOT, seed=SEED,
                                    label="concurrent | published OOF recalibration")], ignore_index=True)
R1.to_csv(REV / "R1_preonset_calibration.csv", index=False)
print(ru.wide(R1).to_string())

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for i, H in enumerate([0, 6, 12]):
    te = WS[H][WS[H].is_test]
    q = pd.qcut(te.p_cal, 10, duplicates="drop")
    tab = te.groupby(q, observed=True).agg(pred=("p_cal", "mean"), obs=("y", "mean"))
    ax[i].plot([0, 1], [0, 1], "--", color="grey", lw=1); ax[i].plot(tab.pred, tab.obs, "o-")
    ax[i].set(title=f"Pre-onset, {H} h before ventilation", xlabel="Predicted", ylabel="Observed")
plt.tight_layout(); plt.savefig(FIGR / "rf_preonset_reliability.png", dpi=200); plt.close()

                                                           AUROC                   AUPRC                   Brier                       CITL                   Slope                     E/O
model                                                                                                                                                                                      
 0 h | raw window peak                    0.651 (0.644 to 0.660)  0.651 (0.640 to 0.662)  0.423 (0.416 to 0.429)     5.453 (5.379 to 5.538)  0.147 (0.139 to 0.157)  0.181 (0.173 to 0.189)
 0 h | recalibrated (Platt, train)        0.651 (0.644 to 0.660)  0.651 (0.640 to 0.662)  0.238 (0.237 to 0.240)    0.006 (-0.027 to 0.036)  0.901 (0.825 to 0.982)  0.997 (0.983 to 1.013)
 2 h | raw window peak                    0.638 (0.627 to 0.649)  0.430 (0.416 to 0.448)  0.271 (0.264 to 0.279)     3.955 (3.854 to 4.048)  0.142 (0.131 to 0.153)  0.217 (0.205 to 0.229)
 2 h | recalibrated (Platt, train)        0.638 (0.627 to 0.

In [7]:
# -------------- R2 · operating characteristics and alert burden (pre-specified thresholds)
rows = []
for H in [0, 6, 12, 24]:
    d = WS[H]; tr, te = d[~d.is_test], d[d.is_test]
    rules = [("raw peak >= 0.5 (published alert)", "score", 0.5), ("raw peak >= 0.7", "score", 0.7),
             ("raw peak >= 0.9", "score", 0.9),
             ("90% specificity (train)", "p_cal", ru.threshold_for_specificity(tr.y, tr.p_cal, 0.90)),
             ("80% sensitivity (train)", "p_cal", ru.threshold_for_sensitivity(tr.y, tr.p_cal, 0.80))]
    for nm, col, thr in rules:
        r = ru.confusion_at(te.y, te[col], thr); r.update(horizon_h=H, rule=nm, prevalence=te.y.mean()); rows.append(r)
R2a = pd.DataFrame(rows)


_g = W.loc[IS_NEG].groupby(["patient_id", "stay_id"])["tmin"].agg(["min", "max"])
_NEG_STAY_SPAN = _g["max"] - _g["min"] + 5.0              # minutes monitored per never-RF stay


def alert_burden(fp, thr, merge_min=60, max_gap_min=MAX_FFILL_MIN):
    """False alert burden in never-RF patients over their monitored time.
    Monitored time = first to last charted window of each ICU stay (rows exist only where
    something was charted, so counting rows would understate time).
    Episode = run of consecutive alerting rows within one stay; a new run starts after a
    non-alerting row, a new stay, or a charting gap > max_gap_min (the pipeline's 4 h
    carry-forward cap). Merged view joins episodes separated by < merge_min minutes."""
    m = IS_NEG
    a = fp[m] >= thr; t = W["tmin"].to_numpy()[m]; p = ROW_PAT[m]
    sid = W["stay_id"].to_numpy()[m]
    newseg = np.r_[True, (p[1:] != p[:-1]) | (sid[1:] != sid[:-1]) | (np.diff(t) > max_gap_min)]
    start = a & ~(np.r_[False, a[:-1]] & ~newseg)
    pdays = float(_NEG_STAY_SPAN.sum()) / 1440.0
    st_t, st_p = t[start], p[start]
    end_idx = np.nonzero(a & ~(np.r_[a[1:], False] & ~np.r_[newseg[1:], True]))[0]
    en_t = t[end_idx]
    gap = np.r_[np.inf, st_t[1:] - en_t[:-1]]
    merged = (st_p != np.r_[-1, st_p[:-1]]) | (gap >= merge_min)
    return dict(threshold=thr, never_rf_patient_days=pdays, alert_window_fraction=float(a.mean()),
                false_alert_episodes_per_patient_day=start.sum() / pdays,
                false_alert_episodes_per_patient_day_merged60=merged.sum() / pdays,
                never_rf_patients_with_any_alert=float(pd.Series(a).groupby(p).any().mean()))


def event_capture(fp, thr, look_h=24):
    """Ventilated patients: alert at any time in the `look_h` hours before first ventilation."""
    m = IS_POS & (MB > 0) & (MB <= look_h * 60)
    a = fp[m] >= thr; p = ROW_PAT[m]; mb = MB[m]
    cap = pd.Series(a).groupby(p).any()
    lead = pd.Series(np.where(a, mb, np.nan)).groupby(p).max() / 60.0
    return dict(ventilated_patients=int(len(cap)), captured_within_24h=float(cap.mean()),
                median_lead_h_first_alert=float(lead.median()))


rows = []
for name in ["Full model", "No failure momentum"]:
    fp = FP[name] if name in FP else run_variant(name)
    FP[name] = fp
    for thr in [0.5, 0.7, 0.9]:
        r = alert_burden(fp, thr); r.update(event_capture(fp, thr)); r["model"] = name; rows.append(r)
R2b = pd.DataFrame(rows)
R2a.to_csv(REV / "R2_operating_characteristics_preonset.csv", index=False)
R2b.to_csv(REV / "R2_alert_burden_continuous.csv", index=False)
print(R2a[["horizon_h", "rule", "threshold", "sensitivity", "specificity", "PPV", "NPV", "prevalence"]].round(3).to_string(index=False))
print(R2b.round(3).to_string(index=False))

 horizon_h                              rule  threshold  sensitivity  specificity   PPV   NPV  prevalence
         0 raw peak >= 0.5 (published alert)      0.500        0.113        0.975 0.822 0.523       0.501
         0                   raw peak >= 0.7      0.700        0.094        0.980 0.829 0.519       0.501
         0                   raw peak >= 0.9      0.900        0.071        0.987 0.842 0.514       0.501
         0           90% specificity (train)      0.494        0.267        0.896 0.720 0.549       0.501
         0           80% sensitivity (train)      0.452        0.800        0.438 0.588 0.686       0.501
         6 raw peak >= 0.5 (published alert)      0.500        0.103        0.975 0.423 0.862       0.148
         6                   raw peak >= 0.7      0.700        0.075        0.980 0.402 0.859       0.148
         6                   raw peak >= 0.9      0.900        0.050        0.987 0.394 0.856       0.148
         6           90% specificity (train)  

In [8]:
# ---------------------------------------------------------------- R3 · ablation
rows, deltas = [], []
for name in VARIANTS:
    t0 = time.time()
    if name not in FP: FP[name] = run_variant(name)
    for H in [0, 6, 12, 24]:
        d = window_scores(FP[name], H)
        rows.append(ru.evaluate(d.y, d.score, d.patient_id, n_boot=N_BOOT, seed=SEED, label=f"{H:>2} h | {name}"))
        if name != "Full model":
            base = WS[H].set_index("patient_id").score.reindex(d.patient_id).to_numpy()
            dd, lo, hi = ru.paired_delta_auc(d.y, d.score.to_numpy(), base, d.patient_id, n_boot=N_BOOT, seed=SEED)
            deltas.append(dict(horizon_h=H, model=name, delta_auroc_vs_full=dd, lo=lo, hi=hi))
    conc = pd.DataFrame({"pid": PAT, "peak": pd.Series(FP[name]).groupby(ROW_PAT).max().to_numpy(),
                         "y": W.is_resp_failure.to_numpy()[STARTS]})
    rows.append(ru.evaluate(conc.y, conc.peak, conc.pid, n_boot=N_BOOT, seed=SEED, label=f"concurrent | {name}"))
    print(f"  {name:<32} done ({time.time()-t0:.0f} s)")
R3 = pd.concat(rows, ignore_index=True); R3d = pd.DataFrame(deltas)
R3.to_csv(REV / "R3_ablation.csv", index=False); R3d.to_csv(REV / "R3_ablation_delta.csv", index=False)
print(ru.wide(R3)[["AUROC", "AUPRC"]].to_string())
print(R3d.round(3).to_string(index=False))

  Full model                       done (11 s)
  No failure momentum              done (12 s)
  Equal feature weights            done (14 s)
  Unordered transitions            done (15 s)
  No age-specific prior/dynamics   done (15 s)
  No temporal dynamics (static)    done (14 s)
                                                              AUROC                   AUPRC
model                                                                                      
 0 h | Full model                            0.650 (0.646 to 0.655)  0.654 (0.647 to 0.660)
 6 h | Full model                            0.659 (0.651 to 0.667)  0.274 (0.264 to 0.286)
12 h | Full model                            0.645 (0.636 to 0.656)  0.197 (0.188 to 0.207)
24 h | Full model                            0.628 (0.616 to 0.640)  0.130 (0.122 to 0.141)
concurrent | Full model                      0.725 (0.721 to 0.728)  0.750 (0.745 to 0.755)
 0 h | No failure momentum                   0.650 (0.646 to 0.655)  0.653

In [9]:
# --------------------------- R4 · baselines on identical windows and patient split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
try:
    import xgboost as xgb; HAVE_XGB = True
except Exception:
    HAVE_XGB = False
try:
    import torch; HAVE_TORCH = True
except Exception:
    HAVE_TORCH = False
print(f"xgboost: {HAVE_XGB} | torch (GRU): {HAVE_TORCH}")


_NEG_CACHE = {}


def _agg(rr, tag):
    g = W.iloc[rr][["patient_id"] + FEATS].groupby("patient_id", sort=False)
    parts = [g[FEATS].mean().add_suffix(f"_mean{tag}"), g[FEATS].min().add_suffix(f"_min{tag}"),
             g[FEATS].max().add_suffix(f"_max{tag}"), g[FEATS].last().add_suffix(f"_last{tag}"),
             (g[FEATS].last() - g[FEATS].first()).add_suffix(f"_delta{tag}"),
             g[FEATS].count().add_suffix(f"_n{tag}")]
    return pd.concat(parts, axis=1).astype(np.float32)


def summaries(H, width, tag):
    """Window summaries (mean, min, max, last, last-first, count) per patient; the negatives'
    windows do not depend on H, so they are computed once per width and cached."""
    pr, nr = window_rows(H, width)
    if width not in _NEG_CACHE:
        _NEG_CACHE[width] = _agg(nr, tag)
    return pd.concat([_agg(pr, tag), _NEG_CACHE[width]])


def gru_scores(H, fit_ids, cal_ids, test_ids, epochs=8, hidden=32):
    """GRU over the 4 h window as 49 five-minute steps of 8 values + 8 observation masks.
    Trained on `fit_ids`, recalibrated on held-out training patients `cal_ids`."""
    torch.manual_seed(SEED)
    pr, nr = window_rows(H)
    rr = np.r_[pr, nr]
    tm = W["tmin"].to_numpy()[rr]
    end = np.where(IS_POS[rr], tm + MB[rr] - H * 60, PT[rr])          # window end = prediction time
    step = np.clip(48 - np.round((end - tm) / 5.0).astype(int), 0, 48)
    ids = W.patient_id.to_numpy()[rr]
    uid, inv = np.unique(ids, return_inverse=True)
    X = W[FEATS].to_numpy(np.float32)[rr]
    fitm = np.isin(ids, np.asarray(fit_ids))
    mu = np.nanmean(X[fitm], 0); sd = np.nanstd(X[fitm], 0) + 1e-6
    Z = np.zeros((len(uid), 49, 2 * len(FEATS)), np.float32)
    Z[inv, step, :len(FEATS)] = np.nan_to_num((X - mu) / sd)
    Z[inv, step, len(FEATS):] = ~np.isnan(X)
    y = np.isin(uid, W.patient_id.to_numpy()[pr]).astype(np.float32)
    pos = pd.Series(np.arange(len(uid)), index=uid)
    ifit, ical, ite = [pos.reindex(np.asarray(v)).dropna().astype(int).to_numpy() for v in (fit_ids, cal_ids, test_ids)]
    net = torch.nn.GRU(2 * len(FEATS), hidden, batch_first=True); head = torch.nn.Linear(hidden, 1)
    opt = torch.optim.Adam(list(net.parameters()) + list(head.parameters()), 1e-3)
    lossf = torch.nn.BCEWithLogitsLoss(); rng = np.random.default_rng(SEED)
    for ep in range(epochs):
        for b in np.array_split(rng.permutation(ifit), max(1, len(ifit) // 512)):
            opt.zero_grad()
            l = lossf(head(net(torch.from_numpy(Z[b]))[1][-1]).squeeze(-1), torch.from_numpy(y[b]))
            l.backward(); opt.step()
    with torch.no_grad():
        f = lambda idx: torch.sigmoid(head(net(torch.from_numpy(Z[idx]))[1][-1]).squeeze(-1)).numpy()
        pc, pt = f(ical), f(ite)
    beta = ru.platt_fit(lg(pc), y[ical])
    return pd.Series(pt, index=uid[ite]), pd.Series(ru.platt_apply(beta, lg(pt)), index=uid[ite])


from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.base import clone
lg = lambda v: np.log(np.clip(v, 1e-6, 1 - 1e-6) / (1 - np.clip(v, 1e-6, 1 - 1e-6)))


def fit_calibrated(mdl, Xtr, ytr, Xte):
    """Fit on training patients; Platt map learned on 5-fold cross-fitted training predictions
    (no in-sample optimism); returns (raw test probability, recalibrated test probability)."""
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    oof = cross_val_predict(clone(mdl), Xtr, ytr, cv=cv, method="predict_proba")[:, 1]
    mdl.fit(Xtr, ytr)
    raw = mdl.predict_proba(Xte)[:, 1]
    return raw, ru.platt_apply(ru.platt_fit(lg(oof), ytr), lg(raw))


def models():
    m = {"Logistic (4 h window summaries)": ("win", make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                              LogisticRegression(max_iter=2000, class_weight="balanced"))),
         "Logistic (4 h + 24 h history)": ("hist", make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                                            StandardScaler(), LogisticRegression(max_iter=3000))),
         "Random forest (4 h + 24 h history)": ("hist", make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                                                RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                                                       n_jobs=-1, random_state=SEED)))}
    if HAVE_XGB:
        m["XGBoost (4 h + 24 h history)"] = ("hist", xgb.XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05,
                                              subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                                              n_jobs=-1, random_state=SEED, eval_metric="logloss"))
    else:
        m["Gradient boosting (4 h + 24 h history)"] = ("hist", HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.05, early_stopping=False, random_state=SEED))
    return m


rows, deltas, TIMES, FEATSETS = [], [], [], {}
for H in [0, 2, 4, 6, 12, 24]:
    d = WS[H].set_index("patient_id")
    Fw = summaries(H, W_MIN, "4h"); Fh = summaries(H, 24 * 60, "24h")
    FEATSETS[H] = (Fw, Fh)
    tr_ids = d.index[~d.is_test]; te_ids = d.index[d.is_test]
    preds = {}
    for name, (fs, mdl) in models().items():
        X = Fw.reindex(d.index) if fs == "win" else pd.concat([Fw, Fh], axis=1).reindex(d.index)
        t0 = time.time()
        preds[name] = fit_calibrated(mdl, X.loc[tr_ids].to_numpy(np.float32), d.loc[tr_ids, "y"].to_numpy(),
                                     X.loc[te_ids].to_numpy(np.float32))
        TIMES.append(dict(horizon_h=H, model=name, fit_and_crossfit_s=time.time() - t0))
    if HAVE_TORCH:
        _p = np.random.default_rng(SEED).permutation(np.asarray(tr_ids))
        _cal = _p[: int(0.2 * len(_p))]; _fit = _p[int(0.2 * len(_p)):]
        try:
            raw, cal = gru_scores(H, _fit, _cal, te_ids)
            preds["GRU (4 h sequence)"] = (raw.reindex(te_ids).to_numpy(), cal.reindex(te_ids).to_numpy())
        except Exception as ex:                       # optional model: never block the other results
            print(f"  GRU skipped at H={H}: {type(ex).__name__}: {ex}")
    yte = d.loc[te_ids, "y"].to_numpy(); belief = d.loc[te_ids, "score"].to_numpy()
    rows.append(ru.evaluate(yte, d.loc[te_ids, "p_cal"], te_ids, n_boot=N_BOOT, seed=SEED,
                            label=f"{H:>2} h | Belief state", rank_score=belief))
    for name, (raw, cal) in preds.items():
        rows.append(ru.evaluate(yte, cal, te_ids, n_boot=N_BOOT, seed=SEED, label=f"{H:>2} h | {name}", rank_score=raw))
        dd, lo, hi = ru.paired_delta_auc(yte, raw, belief, te_ids, n_boot=N_BOOT, seed=SEED)
        deltas.append(dict(horizon_h=H, model=name, delta_auroc_vs_belief=dd, lo=lo, hi=hi))
    print(f"  H={H:>2} h done")
R4 = pd.concat(rows, ignore_index=True); R4d = pd.DataFrame(deltas)
R4.to_csv(REV / "R4_baselines.csv", index=False); R4d.to_csv(REV / "R4_baselines_delta.csv", index=False)
pd.DataFrame(TIMES).to_csv(REV / "R4_baselines_runtime.csv", index=False)
print(ru.wide(R4)[["AUROC", "AUPRC", "Brier", "Slope"]].to_string())
print(R4d.round(3).to_string(index=False))

xgboost: True | torch (GRU): True
  H= 0 h done
  H= 2 h done
  H= 4 h done
  H= 6 h done
  H=12 h done
  H=24 h done
                                                            AUROC                   AUPRC                   Brier                   Slope
model                                                                                                                                    
 0 h | Belief state                        0.651 (0.644 to 0.660)  0.651 (0.640 to 0.662)  0.238 (0.237 to 0.240)  0.901 (0.825 to 0.982)
 0 h | Logistic (4 h window summaries)     0.916 (0.911 to 0.920)  0.927 (0.921 to 0.932)  0.107 (0.105 to 0.110)  1.011 (0.976 to 1.051)
 0 h | Logistic (4 h + 24 h history)       0.938 (0.934 to 0.941)  0.945 (0.940 to 0.948)  0.092 (0.089 to 0.095)  1.019 (0.982 to 1.055)
 0 h | Random forest (4 h + 24 h history)  0.967 (0.964 to 0.969)  0.969 (0.967 to 0.971)  0.070 (0.067 to 0.072)  0.988 (0.960 to 1.018)
 0 h | XGBoost (4 h + 24 h history)        0.973 (0.97

In [10]:
# ---------------------------------- R5 · temporal validation (anchor_year_group)
def _find_patients_csv(*dirs):
    """MIMIC-IV hosp/patients as .csv or .csv.gz; MIMIC_HOSP env var overrides the search."""
    cands = [os.environ.get("MIMIC_HOSP", "")] + [str(d) for d in dirs] + [
        "/Users/sumiyasayeed/Documents/mimic4/mimiciv/hosp", "/Users/sumiyasayeed/Documents/mimic4/hosp"]
    for d in cands:
        for ext in (".csv", ".csv.gz"):
            if d and (Path(d) / f"patients{ext}").exists():
                return Path(d) / f"patients{ext}"
    hits = sorted(Path("/Users/sumiyasayeed/Documents/mimic4").glob("**/patients.csv*"))
    return hits[0] if hits else None


_pc = None if SYNTHETIC_TEST else _find_patients_csv(MIMIC_HOSP_DIR)
if SYNTHETIC_TEST:
    AYG = pd.Series(np.random.default_rng(1).choice(["2008 - 2010", "2011 - 2013", "2014 - 2016", "2017 - 2019"],
                                                    len(PAT)), index=W.subject_id.to_numpy()[STARTS])
elif _pc is not None:
    print("patients table:", _pc)
    AYG = ru.anchor_year_groups(_pc)
if not SYNTHETIC_TEST and _pc is None:
    R5 = None
    print("patients.csv(.gz) not found. Run  os.environ['MIMIC_HOSP'] = '/path/to/mimiciv/hosp'  and rerun this cell.")
else:
    AYG_P = pd.Series(W.subject_id.to_numpy()[STARTS], index=PAT).map(AYG)
    DEV_G, TEST_G = ru.temporal_split(AYG_P, 0.10)
    print("development:", DEV_G, "| temporal test:", TEST_G)
    rows = []
    for H in [0, 6, 12]:
        d = WS[H].set_index("patient_id"); d["ayg"] = AYG_P.reindex(d.index)
        dev, tt = d[d.ayg.isin(DEV_G)], d[d.ayg.isin(TEST_G)]
        beta = ru.platt_fit(dev.score, dev.y)
        rows.append(ru.evaluate(tt.y, ru.platt_apply(beta, tt.score), tt.index, n_boot=N_BOOT, seed=SEED,
                                label=f"{H:>2} h | Belief state (Platt on development years)", rank_score=tt.score))
        Fw, Fh = FEATSETS[H]
        for name, (fs, mdl) in models().items():
            if name.startswith("Random forest"): continue
            X = Fw.reindex(d.index) if fs == "win" else pd.concat([Fw, Fh], axis=1).reindex(d.index)
            pte, cal = fit_calibrated(mdl, X.loc[dev.index].to_numpy(np.float32), dev.y.to_numpy(),
                                      X.loc[tt.index].to_numpy(np.float32))
            rows.append(ru.evaluate(tt.y, cal, tt.index, n_boot=N_BOOT, seed=SEED, label=f"{H:>2} h | {name}", rank_score=pte))
    R5 = pd.concat(rows, ignore_index=True)
    R5["dev_groups"] = "; ".join(DEV_G); R5["test_groups"] = "; ".join(TEST_G)
    R5.to_csv(REV / "R5_temporal_validation.csv", index=False)
    print(ru.wide(R5).to_string())

patients table: /Users/sumiyasayeed/Documents/mimic4/mimiciv/hosp/patients.csv.gz
development: ['2008 - 2010', '2011 - 2013', '2014 - 2016', '2017 - 2019'] | temporal test: ['2020 - 2022']
                                                                   AUROC                   AUPRC                   Brier                       CITL                   Slope                     E/O
model                                                                                                                                                                                              
 0 h | Belief state (Platt on development years)  0.708 (0.696 to 0.723)  0.480 (0.455 to 0.505)  0.244 (0.242 to 0.247)  -1.132 (-1.193 to -1.079)  1.119 (0.993 to 1.257)  1.926 (1.853 to 2.004)
 0 h | Logistic (4 h window summaries)            0.879 (0.867 to 0.889)  0.745 (0.722 to 0.766)  0.135 (0.130 to 0.141)  -1.335 (-1.444 to -1.233)  0.624 (0.582 to 0.668)  1.508 (1.463 to 1.564)
 0 h | Logistic (4 h + 24 h

In [11]:
# ------------------------------------------- R6 · attribution fidelity and stability
# The published explanation names the feature with the largest |LLR(Failure vs Stable)| at
# windows where the failure probability rises by > 0.05. Deletion test: remove one feature
# from that window's update (previous belief and transition kept) and record the change in
# failure probability. Faithful attribution => the named feature is the one whose removal
# lowers the failure probability most.
fp = B_FULL[:, 2]
first = np.zeros(len(W), bool); first[STARTS] = True
dfp = np.r_[0.0, np.diff(fp)]; dfp[first] = 0.0
ev = np.nonzero((dfp > 0.05) & ~first)[0]
ev = np.random.default_rng(SEED).choice(ev, size=min(50_000, len(ev)), replace=False)
prev = B_FULL[ev - 1]
Tb = TBANK[ROW_PAT[ev]].copy()
mom = prev[:, 2] > 0.4; Tb[mom, 2, :] = MOM_ROW
pred = np.einsum("nk,nkj->nj", prev, Tb)
log2pi = np.log(2 * np.pi)
FL = np.zeros((len(ev), 3, len(FEATS))); OBS = np.zeros((len(ev), len(FEATS)), bool)
FLn = np.zeros_like(FL)
rng = np.random.default_rng(SEED)
for j, f in enumerate(FEATS):
    v = W[f].to_numpy(float)[ev]; ok = ~np.isnan(v); OBS[:, j] = ok
    mu, sd = GLOBAL_STATS[f]; z = (v - mu) / sd; zn = z + rng.normal(0, 0.05, len(z))   # 5% of population SD
    for s in range(3):
        loc, sc = STATE_Z[f][s], STATE_SCALE[s]
        FL[ok, s, j] = (FEATURE_WEIGHTS[f] * (-0.5 * ((z - loc) / sc) ** 2 - np.log(sc) - 0.5 * log2pi))[ok]
        FLn[ok, s, j] = (FEATURE_WEIGHTS[f] * (-0.5 * ((zn - loc) / sc) ** 2 - np.log(sc) - 0.5 * log2pi))[ok]


def post_fail(logE):
    lp = np.log(pred + 1e-300) + logE; lp -= lp.max(1, keepdims=True); p = np.exp(lp)
    return (p / p.sum(1, keepdims=True))[:, 2]


recon_err = float(np.max(np.abs(post_fail(FL.sum(2)) - fp[ev])))
DEL = np.stack([fp[ev] - post_fail(FL.sum(2) - FL[:, :, j]) for j in range(len(FEATS))], 1)
LLR = FL[:, 2, :] - FL[:, 0, :]
LLRn = FLn[:, 2, :] - FLn[:, 0, :]
use = OBS.sum(1) >= 2
pick = lambda A: np.where(OBS, A, -np.inf).argmax(1)
top_abs, top_signed, top_del = pick(np.abs(LLR)), pick(LLR), pick(DEL)
from scipy.stats import spearmanr
rho = [spearmanr(LLR[i, OBS[i]], DEL[i, OBS[i]]).correlation for i in np.nonzero(OBS.sum(1) >= 3)[0][:20000]]
R6 = pd.DataFrame([
    dict(check="max |reconstructed - engine failure probability| (exact identity)", value=recon_err),
    dict(check="explanation windows sampled (rise > 0.05)", value=len(ev)),
    dict(check="windows with >= 2 observed features", value=int(use.sum())),
    dict(check="published rule (max |LLR|) = feature whose deletion lowers risk most", value=float((top_abs == top_del)[use].mean())),
    dict(check="signed rule (max LLR toward Failure) = same", value=float((top_signed == top_del)[use].mean())),
    dict(check="median Spearman(signed LLR, deletion effect), windows with >= 3 features", value=float(np.nanmedian(rho))),
    dict(check="sign agreement signed LLR vs deletion effect (|effect| > 0.001)",
         value=float((np.sign(LLR) == np.sign(DEL))[OBS & (np.abs(DEL) > 1e-3)].mean())),
    dict(check="published top driver unchanged under 5% measurement noise", value=float((pick(np.abs(LLRn)) == top_abs)[use].mean())),
    dict(check="signed top driver unchanged under 5% measurement noise", value=float((pick(LLRn) == top_signed)[use].mean())),
])
R6.to_csv(REV / "R6_attribution_fidelity.csv", index=False)
print(R6.to_string(index=False))

                                                                   check        value
       max |reconstructed - engine failure probability| (exact identity) 1.000089e-12
                               explanation windows sampled (rise > 0.05) 5.000000e+04
                                     windows with >= 2 observed features 3.612800e+04
    published rule (max |LLR|) = feature whose deletion lowers risk most 9.276738e-01
                             signed rule (max LLR toward Failure) = same 9.320195e-01
median Spearman(signed LLR, deletion effect), windows with >= 3 features 1.000000e+00
         sign agreement signed LLR vs deletion effect (|effect| > 0.001) 8.656271e-01
               published top driver unchanged under 5% measurement noise 9.902292e-01
                  signed top driver unchanged under 5% measurement noise 9.912256e-01


In [12]:
# ----------------------------------------- R7 · cost and reproducibility record
n_hand = (sum(len(v) for v in STATE_Z.values()) + len(STATE_SCALE) + len(FEATURE_WEIGHTS)
          + 2 * len(GLOBAL_STATS) + 4 * (9 + 3) + 3 + 1)
R7 = dict(numpy_filter_seconds_all_windows=T_NUMPY, windows=int(len(W)), patients=int(len(PAT)),
          published_loop_us_per_window=1e6 * T_LOOP_PER_WIN, numpy_vs_published_max_abs_diff=MAXDIFF,
          engine_parameters_hand_specified=int(n_hand), split_seed=SEED, bootstrap_replicates=N_BOOT,
          environment=ru.env_versions())
json.dump(R7, open(REV / "R7_runtime_and_environment.json", "w"), indent=2, default=str)
print(json.dumps(R7, indent=2, default=str))
print("\nAll revision outputs written to", REV.resolve())

{
  "numpy_filter_seconds_all_windows": 4.980116844177246,
  "windows": 9987192,
  "patients": 65363,
  "published_loop_us_per_window": 8.187931089394745,
  "numpy_vs_published_max_abs_diff": 0.0,
  "engine_parameters_hand_specified": 103,
  "split_seed": 0,
  "bootstrap_replicates": 200,
  "environment": {
    "python": "3.13.5",
    "platform": "macOS-27.0.1-arm64-arm-64bit-Mach-O",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "scipy": "1.18.1",
    "sklearn": "1.9.1",
    "jax": "0.10.0",
    "xgboost": "3.2.0",
    "torch": "2.13.0",
    "lifelines": "0.30.3"
  }
}

All revision outputs written to /Users/sumiyasayeed/Downloads/MSc/PEMDT - Diseases/JBI/PEMDT - Respiratory/results/revision_v2
